In [9]:
import pandas as pd
import optuna
import numpy as np


In [ ]:
df_btc = (
    pd.read_csv("BTC_USDT_15m_1769129842087_1771721842087.csv")
    .set_index("datetime")
    .dropna()
)
df_etc = (
    pd.read_csv("ETC_USDT_15m_1769130073376_1771722073376.csv")
    .set_index("datetime")
    .dropna()
)
df_sol = (
    pd.read_csv("SOL_USDT_15m_1769130125699_1771722125699.csv")
    .set_index("datetime")
    .dropna()
)

display(df_btc.head())
display(df_etc.head())
display(df_sol.head())

,open,high,low,close,volume
datetime,,,,,
2026-01-23 01:00:00,89518.83,89675.77,89518.83,89596.86,37.345470
2026-01-23 01:15:00,89596.86,89690.60,89593.29,89690.60,28.878564
2026-01-23 01:30:00,89690.60,89860.00,89640.82,89820.00,45.025629
2026-01-23 01:45:00,89820.00,89937.50,89766.00,89932.68,39.038629
2026-01-23 02:00:00,89932.68,89975.90,89793.48,89949.90,56.795418


,open,high,low,close,volume
datetime,,,,,
2026-01-23 01:15:00,11.63,11.65,11.63,11.65,19.8245
2026-01-23 01:30:00,11.65,11.69,11.65,11.67,245.9548
2026-01-23 01:45:00,11.67,11.68,11.67,11.68,55.2469
2026-01-23 02:00:00,11.68,11.70,11.68,11.70,19.2683
2026-01-23 02:15:00,11.70,11.70,11.68,11.68,3.9303


,open,high,low,close,volume
datetime,,,,,
2026-01-23 01:15:00,128.32,128.57,128.27,128.42,2342.0048
2026-01-23 01:30:00,128.42,128.78,128.33,128.71,3278.7293
2026-01-23 01:45:00,128.71,128.92,128.63,128.90,2657.2998
2026-01-23 02:00:00,128.90,128.96,128.56,128.78,3024.3740
2026-01-23 02:15:00,128.78,128.81,128.40,128.50,484.7823


In [10]:
def backtest(
    df: pd.DataFrame,
    window_rolling_mean: int,
    window_rolling_std: int,
    zscore: float,
    length_rsi: int = 14,
    overbought: float = 70,
    oversold: float = 30,
    adx: float = 20,
    adxlen: int = 14,
) -> dict:
    df_bt = df.copy()
    df_bt.dropna(inplace=True)

    df_bt["mean"] = df_bt["close"].rolling(window=window_rolling_mean).mean()
    df_bt["std"] = df_bt["close"].rolling(window=window_rolling_std).std()
    df_bt["zscore"] = (df_bt["close"] - df_bt["mean"]) / df_bt["std"]

    # --- RSI Calculation ---
    change = df_bt["close"].diff()
    gain = change.clip(lower=0.0)
    loss = -change.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1 / length_rsi, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / length_rsi, adjust=False).mean()
    rs = avg_gain / avg_loss
    df_bt["rsi"] = 100 - (100 / (1 + rs))

    # --- ADX Calculation ---
    tr1 = df_bt["high"] - df_bt["low"]
    tr2 = (df_bt["high"] - df_bt["close"].shift(1)).abs()
    tr3 = (df_bt["low"] - df_bt["close"].shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    up = df_bt["high"].diff()
    down = -df_bt["low"].diff()

    plusDM = up.where((up > down) & (up > 0), 0.0)
    plusDM.iloc[0] = np.nan
    minusDM = down.where((down > up) & (down > 0), 0.0)
    minusDM.iloc[0] = np.nan

    truerange = tr.ewm(alpha=1 / adxlen, adjust=False).mean()
    plus = (100 * plusDM.ewm(alpha=1 / adxlen, adjust=False).mean() / truerange).ffill()
    minus = (
        100 * minusDM.ewm(alpha=1 / adxlen, adjust=False).mean() / truerange
    ).ffill()

    sum_dm = plus + minus
    sum_dm = sum_dm.replace(0, 1)  # prevent division by zero
    dx = 100 * (plus - minus).abs() / sum_dm
    df_bt["adx"] = dx.ewm(alpha=1 / adxlen, adjust=False).mean()

    # --- Trading Logic ---
    df_bt["signal"] = np.nan

    # Short condition
    short_cond = (
        (df_bt["zscore"] > zscore) & (df_bt["rsi"] > overbought) & (df_bt["adx"] < adx)
    )
    df_bt.loc[short_cond, "signal"] = -1

    # Long condition
    long_cond = (
        (df_bt["zscore"] < -zscore) & (df_bt["rsi"] < oversold) & (df_bt["adx"] < adx)
    )
    df_bt.loc[long_cond, "signal"] = 1

    # Exit condition: zscore crosses 0 (reverts to mean)
    exit_cond = df_bt["zscore"] * df_bt["zscore"].shift(1) < 0
    df_bt.loc[exit_cond, "signal"] = 0

    # Hold position
    df_bt["position"] = df_bt["signal"].ffill().fillna(0)

    # --- Performance Metrics ---
    df_bt["returns"] = df_bt["close"].pct_change()

    # Strategy returns (current position was carried from yesterday)
    df_bt["strategy_ret"] = df_bt["position"].shift(1) * df_bt["returns"]

    total_return = (1 + df_bt["strategy_ret"]).prod() - 1

    # Annualized based on 15-minute timeframe
    periods_per_year = 365 * 24 * 4
    std_dev = df_bt["strategy_ret"].std()
    sharpe_ratio = (
        (df_bt["strategy_ret"].mean() / std_dev * np.sqrt(periods_per_year))
        if std_dev > 0
        else 0
    )

    cum_returns = (1 + df_bt["strategy_ret"]).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns - running_max) / running_max
    max_drawdown = drawdown.min()

    winning_trades = len(df_bt[df_bt["strategy_ret"] > 0])
    losing_trades = len(df_bt[df_bt["strategy_ret"] < 0])
    total_trades = winning_trades + losing_trades
    win_rate = (winning_trades / total_trades) if total_trades > 0 else 0

    return {
        "Total Return": total_return,
        "Sharpe Ratio": sharpe_ratio,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
    }


In [11]:
def run_optimization(df: pd.DataFrame) -> dict:
    def objective(trial: optuna.Trial):
        window_rolling_mean = trial.suggest_int("window_rolling_mean", 5, 100)
        window_rolling_std = trial.suggest_int("window_rolling_std", 5, 100)
        zscore = trial.suggest_float("zscore", 1.0, 3.5)
        length_rsi = trial.suggest_int("length_rsi", 5, 100)
        overbought = trial.suggest_int("overbought", 50, 100)
        oversold = trial.suggest_int("oversold", 0, 50)
        adx = trial.suggest_int("adx", 0, 100)
        adx_len = trial.suggest_int("adx_len", 5, 100)

        try:
            result_bt = backtest(
                df=df,
                window_rolling_mean=window_rolling_mean,
                window_rolling_std=window_rolling_std,
                zscore=zscore,
                length_rsi=length_rsi,
                overbought=overbought,
                oversold=oversold,
                adx=adx,
                adxlen=adx_len,
            )

            sharpe = float(result_bt["Sharpe Ratio"])
            if pd.isna(sharpe) or sharpe == float("inf") or sharpe == float("-inf"):
                return 0.0

            return sharpe

        except Exception as e:
            # Si ocurre algún error inesperado de Pandas, descartamos el trial en lugar de parar toda la optimización
            raise optuna.TrialPruned()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=200)

    print("Best parameters:", study.best_params)
    print("Best Sharpe Ratio:", study.best_value)

    return study.best_params


In [16]:
best_params = run_optimization(df_btc)
result_bt_btc = backtest(
    df=df_btc,
    window_rolling_mean=best_params["window_rolling_mean"],
    window_rolling_std=best_params["window_rolling_std"],
    zscore=best_params["zscore"],
    length_rsi=best_params["length_rsi"],
    overbought=best_params["overbought"],
    oversold=best_params["oversold"],
    adx=best_params["adx"],
    adxlen=best_params["adx_len"],
)
display(result_bt_btc)

[I 2026-02-22 03:05:16,509] A new study created in memory with name: no-name-cc9ad52c-e5f3-4cf4-8c11-16cdf6f6cfee
[I 2026-02-22 03:05:16,519] Trial 0 finished with value: 5.650250502141901 and parameters: {'window_rolling_mean': 46, 'window_rolling_std': 70, 'zscore': 1.4588834736915202, 'length_rsi': 78, 'overbought': 55, 'oversold': 2, 'adx': 87, 'adx_len': 93}. Best is trial 0 with value: 5.650250502141901.
[I 2026-02-22 03:05:16,526] Trial 1 finished with value: 2.3675907979926167 and parameters: {'window_rolling_mean': 41, 'window_rolling_std': 88, 'zscore': 1.7472032274520959, 'length_rsi': 7, 'overbought': 62, 'oversold': 9, 'adx': 37, 'adx_len': 65}. Best is trial 0 with value: 5.650250502141901.
[I 2026-02-22 03:05:16,533] Trial 2 finished with value: -1.6100941715562733 and parameters: {'window_rolling_mean': 61, 'window_rolling_std': 98, 'zscore': 2.8480568389158494, 'length_rsi': 22, 'overbought': 71, 'oversold': 28, 'adx': 91, 'adx_len': 18}. Best is trial 0 with value: 5.

Best parameters: {'window_rolling_mean': 42, 'window_rolling_std': 14, 'zscore': 2.4457595889074395, 'length_rsi': 90, 'overbought': 57, 'oversold': 35, 'adx': 75, 'adx_len': 46}
Best Sharpe Ratio: 11.366225415219846


{'Total Return': np.float64(0.1848471298444958),
 'Sharpe Ratio': np.float64(11.366225415219846),
 'Max Drawdown': np.float64(-0.015966794943692263),
 'Win Rate': 0.5888888888888889}

In [17]:
best_params = run_optimization(df_etc)
result_bt_etc = backtest(
    df=df_etc,
    window_rolling_mean=best_params["window_rolling_mean"],
    window_rolling_std=best_params["window_rolling_std"],
    zscore=best_params["zscore"],
    length_rsi=best_params["length_rsi"],
    overbought=best_params["overbought"],
    oversold=best_params["oversold"],
    adx=best_params["adx"],
    adxlen=best_params["adx_len"],
)
display(result_bt_etc)

[I 2026-02-22 03:18:28,496] A new study created in memory with name: no-name-06e4869a-5b8b-4643-9a68-1da7b7a64d03
[I 2026-02-22 03:18:28,505] Trial 0 finished with value: 0.0 and parameters: {'window_rolling_mean': 75, 'window_rolling_std': 86, 'zscore': 2.7857020130495638, 'length_rsi': 97, 'overbought': 71, 'oversold': 25, 'adx': 36, 'adx_len': 57}. Best is trial 0 with value: 0.0.
[I 2026-02-22 03:18:28,512] Trial 1 finished with value: -0.6090813305722289 and parameters: {'window_rolling_mean': 86, 'window_rolling_std': 100, 'zscore': 1.2976495560578005, 'length_rsi': 9, 'overbought': 63, 'oversold': 46, 'adx': 62, 'adx_len': 14}. Best is trial 0 with value: 0.0.
[I 2026-02-22 03:18:28,519] Trial 2 finished with value: 2.9251065825302245 and parameters: {'window_rolling_mean': 17, 'window_rolling_std': 10, 'zscore': 1.3940000153739458, 'length_rsi': 76, 'overbought': 67, 'oversold': 3, 'adx': 87, 'adx_len': 76}. Best is trial 2 with value: 2.9251065825302245.
[I 2026-02-22 03:18:28

Best parameters: {'window_rolling_mean': 97, 'window_rolling_std': 16, 'zscore': 2.5446187017058297, 'length_rsi': 57, 'overbought': 72, 'oversold': 34, 'adx': 66, 'adx_len': 81}
Best Sharpe Ratio: 7.008143008367439


{'Total Return': np.float64(0.2339213159376068),
 'Sharpe Ratio': np.float64(7.008143008367439),
 'Max Drawdown': np.float64(-0.059760956175298384),
 'Win Rate': 0.5736434108527132}

In [18]:
best_params = run_optimization(df_sol)
result_bt_sol = backtest(
    df=df_sol,
    window_rolling_mean=best_params["window_rolling_mean"],
    window_rolling_std=best_params["window_rolling_std"],
    zscore=best_params["zscore"],
    length_rsi=best_params["length_rsi"],
    overbought=best_params["overbought"],
    oversold=best_params["oversold"],
    adx=best_params["adx"],
    adxlen=best_params["adx_len"],
)
display(result_bt_sol)

[I 2026-02-22 03:23:35,138] A new study created in memory with name: no-name-f1ab4dda-8e39-4e8f-bad4-ebda9a5d945a
[I 2026-02-22 03:23:35,144] Trial 0 finished with value: -0.98064842727977 and parameters: {'window_rolling_mean': 47, 'window_rolling_std': 42, 'zscore': 1.9547019775145738, 'length_rsi': 86, 'overbought': 84, 'oversold': 39, 'adx': 44, 'adx_len': 85}. Best is trial 0 with value: -0.98064842727977.
[I 2026-02-22 03:23:35,154] Trial 1 finished with value: -0.8926534658062485 and parameters: {'window_rolling_mean': 57, 'window_rolling_std': 72, 'zscore': 1.0717599831081737, 'length_rsi': 29, 'overbought': 96, 'oversold': 26, 'adx': 16, 'adx_len': 74}. Best is trial 1 with value: -0.8926534658062485.
[I 2026-02-22 03:23:35,162] Trial 2 finished with value: -3.380180003397883 and parameters: {'window_rolling_mean': 97, 'window_rolling_std': 15, 'zscore': 1.2547278777643025, 'length_rsi': 36, 'overbought': 61, 'oversold': 41, 'adx': 87, 'adx_len': 33}. Best is trial 1 with valu

Best parameters: {'window_rolling_mean': 93, 'window_rolling_std': 93, 'zscore': 2.127317125383313, 'length_rsi': 97, 'overbought': 50, 'oversold': 36, 'adx': 100, 'adx_len': 80}
Best Sharpe Ratio: 7.614743269191434


{'Total Return': np.float64(0.3268047850595117),
 'Sharpe Ratio': np.float64(7.614743269191434),
 'Max Drawdown': np.float64(-0.1080290223515071),
 'Win Rate': 0.5288326300984529}